In [1]:
import json
import uuid
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import udahub

# Udahub Application

## Core Database

**Init DB**

In [2]:
udahub_db = "data/core/udahub.db"

In [3]:
reset_db(udahub_db)

✅ Removed existing data/core/udahub.db
2026-05-29 20:11:48,399 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-29 20:11:48,399 INFO sqlalchemy.engine.Engine COMMIT
✅ Recreated data/core/udahub.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{udahub_db}", echo=False)
udahub.Base.metadata.create_all(bind=engine)

**Account**

In [5]:
account_id = "cultpass"
account_name = "CultPass Card"

In [6]:
with get_session(engine) as session:
    account = udahub.Account(
        account_id=account_id,
        account_name=account_name,
    )
    session.add(account)

## Integrations

**Knowledge Base**

In [7]:
# TODO: Create additional 10 articles

In [8]:
cultpass_articles = []

with open('data/external/cultpass_articles.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_articles.append(json.loads(line))

In [9]:
cultpass_articles

[{'title': 'How to Reserve a Spot for an Event',
  'content': 'If a user asks how to reserve an event:\n\n- Guide them to the CultPass app\n- Instruct them to browse the experience catalog and tap \'Reserve\'\n- If it\'s a premium or limited event, check if reservation confirmation is required via email\n- Remind them to arrive at least 15 minutes early with their QR code visible\n\n**Suggested phrasing:**\n"You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping \'Reserve\'. Be sure to arrive 15 minutes early with your QR code ready."',
  'tags': 'reservation, events, booking, attendance'},
 {'title': "What's Included in a CultPass Subscription",
  'content': 'Each user is entitled to 4 cultural experiences per month, which may include:\n- Art exhibitions\n- Museum entries\n- Music concerts\n- Film screenings and more\n\nSome premium experiences may require an additional fee (visible in the app).\n\n**Suggested phrasing:**\n"Your CultPass s

In [10]:
if len(cultpass_articles) < 14:
    raise AssertionError("You should load the articles with at least 14 records")

In [11]:
with get_session(engine) as session:
    kb = []
    for article in cultpass_articles:
        knowledge = udahub.Knowledge(
            article_id=str(uuid.uuid4()),
            account_id=account_id,
            title=article["title"],
            content=article["content"],
            tags=article["tags"]
        )
        kb.append(knowledge)
    session.add_all(kb) 
    

**Ticket**

In [12]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [13]:
tickets_info = [
    {
        "status": "open",
        "content": "I can't log in to my Cultpass account.",
        "owner_id": cultpass_users[0]["id"],
        "owner_name": cultpass_users[0]["name"],
        "role": "user",
        "channel": "chat",
        "tags": "login, access",
    },
    {
        "status": "open",
        "content": "I was charged twice for my monthly subscription.",
        "owner_id": cultpass_users[1]["id"],
        "owner_name": cultpass_users[1]["name"],
        "role": "user",
        "channel": "email",
        "tags": "billing, duplicate charge",
    },
    {
        "status": "open",
        "content": "My reservation for yoga class disappeared from the app.",
        "owner_id": cultpass_users[2]["id"],
        "owner_name": cultpass_users[2]["name"],
        "role": "user",
        "channel": "chat",
        "tags": "reservation, app issue",
    },
    {
        "status": "open",
        "content": "I need to update the email address on my account.",
        "owner_id": cultpass_users[3]["id"],
        "owner_name": cultpass_users[3]["name"],
        "role": "user",
        "channel": "web",
        "tags": "account, profile update",
    },
    {
        "status": "open",
        "content": "My account says blocked and I don't know why.",
        "owner_id": cultpass_users[4]["id"],
        "owner_name": cultpass_users[4]["name"],
        "role": "user",
        "channel": "chat",
        "tags": "account, blocked",
    },
    {
        "status": "open",
        "content": "I canceled my membership but I still got billed.",
        "owner_id": cultpass_users[5]["id"],
        "owner_name": cultpass_users[5]["name"],
        "role": "user",
        "channel": "email",
        "tags": "billing, cancellation",
    },
    {
        "status": "open",
        "content": "The app crashes every time I try to book a class.",
        "owner_id": cultpass_users[6]["id"],
        "owner_name": cultpass_users[6]["name"],
        "role": "user",
        "channel": "chat",
        "tags": "app issue, booking",
    },
    {
        "status": "open",
        "content": "I forgot my password and the reset link is not working.",
        "owner_id": cultpass_users[7]["id"],
        "owner_name": cultpass_users[7]["name"],
        "role": "user",
        "channel": "web",
        "tags": "password reset, login",
    },
    {
        "status": "open",
        "content": "Can I transfer my reservation to a different time slot?",
        "owner_id": cultpass_users[8]["id"],
        "owner_name": cultpass_users[8]["name"],
        "role": "user",
        "channel": "chat",
        "tags": "reservation, schedule change",
    },
    {
        "status": "open",
        "content": "I was marked as a no-show even though I attended the class.",
        "owner_id": cultpass_users[9]["id"],
        "owner_name": cultpass_users[9]["name"],
        "role": "user",
        "channel": "email",
        "tags": "attendance, no-show",
    },
    {
        "status": "open",
        "content": "The promo code I received does not apply at checkout.",
        "owner_id": cultpass_users[10]["id"],
        "owner_name": cultpass_users[10]["name"],
        "role": "user",
        "channel": "web",
        "tags": "promo code, checkout",
    },
    {
        "status": "open",
        "content": "I want to pause my membership for one month.",
        "owner_id": cultpass_users[11]["id"],
        "owner_name": cultpass_users[11]["name"],
        "role": "user",
        "channel": "chat",
        "tags": "membership, pause",
    },
    {
        "status": "open",
        "content": "I booked a class by mistake and need to cancel it.",
        "owner_id": cultpass_users[12]["id"],
        "owner_name": cultpass_users[12]["name"],
        "role": "user",
        "channel": "chat",
        "tags": "reservation, cancellation",
    },
    {
        "status": "open",
        "content": "My payment method keeps getting declined even though the card works elsewhere.",
        "owner_id": cultpass_users[13]["id"],
        "owner_name": cultpass_users[13]["name"],
        "role": "user",
        "channel": "email",
        "tags": "payment, billing",
    },
    {
        "status": "open",
        "content": "I can't see my active subscription in the account dashboard.",
        "owner_id": cultpass_users[14]["id"],
        "owner_name": cultpass_users[14]["name"],
        "role": "user",
        "channel": "web",
        "tags": "subscription, dashboard",
    },
    {
        "status": "open",
        "content": "The instructor changed but I was not notified before the class.",
        "owner_id": cultpass_users[15]["id"],
        "owner_name": cultpass_users[15]["name"],
        "role": "user",
        "channel": "chat",
        "tags": "class info, notification",
    },
    {
        "status": "open",
        "content": "I need a receipt for my last payment for reimbursement purposes.",
        "owner_id": cultpass_users[16]["id"],
        "owner_name": cultpass_users[16]["name"],
        "role": "user",
        "channel": "email",
        "tags": "receipt, billing",
    },
    {
        "status": "open",
        "content": "The location listed for my class seems incorrect.",
        "owner_id": cultpass_users[17]["id"],
        "owner_name": cultpass_users[17]["name"],
        "role": "user",
        "channel": "web",
        "tags": "location, class info",
    },
    {
        "status": "open",
        "content": "I want to know why my account was flagged for suspicious activity.",
        "owner_id": cultpass_users[18]["id"],
        "owner_name": cultpass_users[18]["name"],
        "role": "user",
        "channel": "chat",
        "tags": "security, account",
    },
    {
        "status": "open",
        "content": "I was billed after my free trial ended without a reminder.",
        "owner_id": cultpass_users[19]["id"],
        "owner_name": cultpass_users[19]["name"],
        "role": "user",
        "channel": "email",
        "tags": "free trial, billing",
    },
]

In [14]:
with get_session(engine) as session:
    for ticket_info in tickets_info:
        user = session.query(udahub.User).filter_by(
            account_id=account_id,
            external_user_id=ticket_info["owner_id"],
        ).first()

        if not user:
            user = udahub.User(
                user_id=str(uuid.uuid4()),
                account_id=account_id,
                external_user_id=ticket_info["owner_id"],
                user_name=ticket_info["owner_name"],
            )
            session.add(user)
            session.flush()

        ticket = udahub.Ticket(
            ticket_id=str(uuid.uuid4()),
            account_id=account_id,
            user_id=user.user_id,
            channel=ticket_info["channel"],
        )
        session.add(ticket)

        metadata = udahub.TicketMetadata(
            ticket_id=ticket.ticket_id,
            status=ticket_info["status"],
            main_issue_type=None,
            tags=ticket_info["tags"],
        )

        first_message = udahub.TicketMessage(
            message_id=str(uuid.uuid4()),
            ticket_id=ticket.ticket_id,
            role=ticket_info["role"],
            content=ticket_info["content"],
        )

        session.add_all([metadata, first_message])


# Tests

In [15]:
import json
from pathlib import Path

output_path = Path("external/data/cultpass_tickets.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f:
    for ticket in tickets_info:
        f.write(json.dumps(ticket) + "\n")

In [16]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    print(account)

<Account(account_id='cultpass', account_name='CultPass Card')>


In [17]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    for article in account.knowledge_articles:
        print(article)

<Knowledge(article_id='cc9d29e7-f947-40bc-951a-77c13bceb91f', title='How to Reserve a Spot for an Event')>
<Knowledge(article_id='db4cbcd0-6374-451c-afc6-06002ad5f941', title='What's Included in a CultPass Subscription')>
<Knowledge(article_id='0fc1ef9a-bc1a-41c9-ad78-16514da18ac1', title='How to Cancel or Pause a Subscription')>
<Knowledge(article_id='e1f7f868-b4a2-43a6-bca4-120d061ed538', title='How to Handle Login Issues?')>
<Knowledge(article_id='8fceff3e-d17a-4c20-8f43-5c4c157dad2d', title='App Crashes on Launch')>
<Knowledge(article_id='eed10500-3038-43f2-b87f-e2bb5370a1da', title='QR Code Not Displaying for Reservation')>
<Knowledge(article_id='62dce5ef-8275-43ae-9932-8fbad7ead50a', title='App Loading Slowly or Freezing')>
<Knowledge(article_id='68949dac-f412-43de-9349-a4d7c6d8cd6b', title='Unable to Log In After Multiple Attempts')>
<Knowledge(article_id='82637e83-892c-4565-a9d3-6499baf2f2de', title='How to Update Payment Method')>
<Knowledge(article_id='8b85064f-d7c8-41c9-b5c8

In [18]:
print(account_id)
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(account_id=account_id).first()
    print(account)

cultpass
<Account(account_id='cultpass', account_name='CultPass Card')>


In [19]:
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    for user in users:
        print(user)

<User(user_id='35325d92-6ddc-49e1-b907-cd5718c9006b', user_name='Alice Kingsley', external_user_id='a4ab87')>
<User(user_id='34e97410-830f-4b65-9d43-52a0708fa76e', user_name='Bob Stone', external_user_id='f556c0')>
<User(user_id='9d1661f6-bfa1-4251-b983-96ddb68bf9db', user_name='Cathy Bloom', external_user_id='88382b')>
<User(user_id='96b20945-139e-41ab-acfc-f6b1024ccee9', user_name='David Noir', external_user_id='888fb2')>
<User(user_id='18aae800-e3a9-485b-8560-fd3f45e94946', user_name='Eva Green', external_user_id='f1f10d')>
<User(user_id='20fc2ebc-4366-41c4-9537-95641e2be627', user_name='Frank Ocean', external_user_id='e6376d')>
<User(user_id='b25700ea-b5d3-4ee3-9491-21d5989080c0', user_name='Grace Hollow', external_user_id='b7c921')>
<User(user_id='95ecd96c-83eb-4a2f-a017-608f4ce5a78d', user_name='Henry Vale', external_user_id='c3d442')>
<User(user_id='46144524-bdb5-430e-ab0f-9c037c98786e', user_name='Isla Frost', external_user_id='d8e113')>
<User(user_id='b4f51028-e247-41fa-a261-1

In [20]:
with get_session(engine) as session:
    tickets = session.query(udahub.Ticket).all()

    for ticket in tickets:
        print(f"Ticket ID: {ticket.ticket_id}")
        print(f"Account ID: {ticket.account_id}")
        print(f"User ID: {ticket.user_id}")
        print(f"Channel: {ticket.channel}")

        for message in ticket.messages:
            print(f"  Message ({message.role}): {message.content}")

        print("-" * 40)

Ticket ID: 3bb297de-496d-4693-89fe-0f51ee18ec57
Account ID: cultpass
User ID: 35325d92-6ddc-49e1-b907-cd5718c9006b
Channel: chat
  Message (RoleEnum.user): I can't log in to my Cultpass account.
----------------------------------------
Ticket ID: c72c624f-eb09-4fa0-a0f8-cae26cc628b3
Account ID: cultpass
User ID: 34e97410-830f-4b65-9d43-52a0708fa76e
Channel: email
  Message (RoleEnum.user): I was charged twice for my monthly subscription.
----------------------------------------
Ticket ID: 73976d25-d4cd-4de9-9f20-2a56e3d7495b
Account ID: cultpass
User ID: 9d1661f6-bfa1-4251-b983-96ddb68bf9db
Channel: chat
  Message (RoleEnum.user): My reservation for yoga class disappeared from the app.
----------------------------------------
Ticket ID: 3abc00c5-840a-4513-b091-02b563d8b40b
Account ID: cultpass
User ID: 96b20945-139e-41ab-acfc-f6b1024ccee9
Channel: web
  Message (RoleEnum.user): I need to update the email address on my account.
----------------------------------------
Ticket ID: 9d3b8ba